<a href="https://colab.research.google.com/github/yaj0212/Introductory-Machine-Learning/blob/main/assignments/assn4/assn4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introductory Machine Learning: Assignment 4

**Deadline:**

Assignment 4 is due Thursday, Apr 9 at 11:59pm. Late work will not be accepted as per the course policies (see the Syllabus and Course policies on [Canvas](https://canvas.yale.edu).

Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. Acknowledge any use of an AI system such as ChatGPT or CoPilot.

You should start early so that you have time to get help if you're stuck. The drop-in office hours schedule can be found on [Canvas](https://canvas.yale.edu).  You can also post questions or start discussions on [Ed Discussion](https://edstem.org/us/courses/91289/discussion). The problems are broken up into steps that should help you to make steady progress.

**Submission:**

Submit your assignment as a .pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each homework assignment are numbered. Note: When submitting on Gradescope, please select the correct pages of your pdf that correspond to each problem. This will allow graders to find your complete solution to each problem.

To produce the .pdf, please do the following in order to preserve the cell structure of the notebook:  
1.  Go to "File" at the top-left of your Jupyter Notebook
2.  Under "Download as", select "HTML (.html)"
3.  After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
4.  From the print window, select the option to save as a .pdf

**Topics**
1. Language models
2. Word embeddings

### Problem 1: Gutenberg Books Language Models (15 points)

For this problem you will process books from the [Project Gutenberg](https://www.gutenberg.org/) site which is a public respository of large numbers of books that are in the public domain. You'll build *character-based* (as opposed to word-based) language models on one book, and predict the letters of the other book using the model.


In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plots
%matplotlib inline

The following helper function `read_url` reads in the text at the given url, and then uses some
[regular expressions](https://www.w3schools.com/python/python_regex.asp) to process the book, removing
everything but the letters a-z, space and period.

In [ ]:
from urllib.request import urlopen
import re

def read_url(url):
    return re.sub(r'\s+', ' ', urlopen(url).read().decode())

def process_text(text):
    text = re.sub(r'[^a-z .]', '', text.lower())
    return re.sub(r'[\[\]\_]', '', text)

<img src="https://www.gutenberg.org/cache/epub/76/pg76.cover.medium.jpg" width="210" align="top">

The online book for "Adventures of Huckleberry Finn," by Mark Twain, is [here](https://www.gutenberg.org/ebooks/76).
From this web site you can see various metadata for the book as well as the [link the text itself](https://www.gutenberg.org/files/76/76-0.txt), which is [https://www.gutenberg.org/files/76/76-0.txt](https://www.gutenberg.org/files/76/76-0.txt)

The book for Mark Twain's "A Connecticut Yankee in King Arthur's Court" is [here](https://www.gutenberg.org/ebooks/86).
In the following cell we read in both of these books, and remove all characters except a-z, space, and period.

In [ ]:
huck_finn_url = 'https://www.gutenberg.org/files/76/76-0.txt'
huck_finn_text_raw = read_url(huck_finn_url)
huck_finn_text = process_text(huck_finn_text_raw)

ct_yankee_url = 'https://www.gutenberg.org/files/86/86-0.txt'
ct_yankee_text_raw = read_url(ct_yankee_url)
ct_yankee_text = process_text(ct_yankee_text_raw)


In [ ]:
print("\nSample of raw text:\n")
print(huck_finn_text_raw[10000:11000])

print("\nSample of processed text:\n")
print(huck_finn_text[10000:11000])

<img src="https://www.gutenberg.org/cache/epub/1342/pg1342.cover.medium.jpg" width="210" align="top">


The online book for "Pride and Prejudice", by Jane Austen, is [here](https://www.gutenberg.org/ebooks/1342).
And [here](https://www.gutenberg.org/ebooks/158) is the online book for Jane Austen's "Emma".  In the following cell we read in both of these books, and remove all characters except a-z, space, and period.



In [ ]:
pride_and_prejudice_url = 'https://www.gutenberg.org/files/1342/1342-0.txt'
pride_and_prejudice_text_raw = read_url(pride_and_prejudice_url)
pride_and_prejudice_text = process_text(pride_and_prejudice_text_raw)


emma_url = 'https://www.gutenberg.org/files/158/158-0.txt'
emma_text_raw = read_url(emma_url)
emma_text = process_text(emma_text_raw)

In [ ]:
print("\nSample of raw text:\n")
print(emma_text_raw[10000:11000])

print("\nSample of processed text:\n")
print(emma_text[10000:11000])

The following cell defines some helper code. You should just run this cell; do not change any of the code.

The first function, `ngrams`, takes some input text and a value of `n`. The function then
iterates over the string and counts the number of occurrences of each substring of `n` characters. This is done with the very handy `Counter` class.

We then define a class `language_model` that is a 4-gram character-based language model. The probability of the "next character" is computed using linear interpolation, as described in class.  A weight is assigned to unigrams, bigrams, trigrams, and four-grams (quadgrams?). The bigram probability that, for example, the letter `t` follows the letter `h` is the count of the bigram `ht` divided by the count of the unigram `h`. We add a little bit (1e-10) to the denominator to avoid dividing by zero.

We return the logarithm of the probability, because this will be convenient when computing perplexities.


In [ ]:
def ngrams(text, n=2):
    return Counter([text[(i-n):i] for i in np.arange(n, len(text)+1)])

class language_model:

    def __init__(self, text):
        self.one = ngrams(text, 1)
        self.two = ngrams(text, 2)
        self.three = ngrams(text, 3)
        self.four = ngrams(text, 4)
        self.weight = [0.1, 0.2, 0.3, 0.4]

    def set_weights(self, weights):
        self.weight = weights / np.sum(weights)

    def log_probability(self, gram):
        numer = [self.one[gram[3:]], self.two[gram[2:]], self.three[gram[1:]], self.four[gram[0:]]]
        denom = [sum(self.one[g] for g in self.one), self.one[gram[2:3]], self.two[gram[1:3]], self.three[gram[0:3]]]
        prob = 0
        for i in np.arange(4):
            prob += self.weight[i] * numer[i] / (denom[i]+1e-10)
        return np.log(prob)



### Problem 1.1

Just to be sure we understand what a character-based language model is, let's write an expression
for the probability in an example. Suppose the language model assigns
weight $w_1 = 0.1$ to the unigram model, weight $w_2 = 0.2$ to the bigram model, weight $w_3 = 0.3$ to the trigram model, and weight $w_4 = .4$ to the four-gram model. Note that we must have $w_1+w_2+w_3+w_4 = 1$.

Write an expression for the probability $p(\mbox{z} \,|\, \mbox{qui})$ that the letter $\mbox{z}$ follows the three letters $\mbox{qui}$. Assume that the unigram, bigram, trigram, and four-gram components are given by ratios of
counts in the training data, as in the code above. For example, the bigram probability would be written as

$$ \frac{\mbox{count}(iz)}{\mbox{count}(i)}$$


**[Your answer here]**

Now, the cell below constructs two language models, one on the text of Jane Austen's "Emma", the other on the text of Mark Twain's "Huckleberry Finn". You will need it for Problem 1.3 - 1.6 later.

In [ ]:
emma_lm = language_model(emma_text)
huck_finn_lm = language_model(huck_finn_text)

### Problem 1.2

In this sub-problem, your job is to write a function that takes a language model `lm`, and a text string `text`, and computes the perplexity of the language model on the text.

Hints:
* Your function can ignore the first three characters of the text. Thus, you can begin predicting the fourth character from the first three.
* Either extract the sequence of 4-character substrings, or make a call to `ngrams(text, n=4)` to get a set of 4-grams and their counts on the text.
* Compute the logarithm of probability of the text. If you compute the probability, you will get a very tiny number and numerical "underflow".
* Use the function `lm.log_probability` where `lm` is a instance of the class `language_model`. For example, `emma_lm.log_probability('emma')` will compute the logarithm of the probability that the character "a" follows the three characters "emm" using the language model computed on Jane Austen's "Emma".
* Once you have the logarithm of the probability of the entire text, you'll need to scale appropriately and then take the exponential, using `np.exp`.
* Work out the formula by "pencil and paper" before trying to write the function.


In [ ]:
def compute_perplexity(text, lm):
  # Your code here
  ...
  return


### Problem 1.3

To test your implementation of the perplexity function, evaluate the followign cell. This
computes the perplexity of the "Emma" language model on all four of the books: "Emma", "Pride and Prejudice", "Huckleberry Finn", and "Connecticut Yankee". For this problem, you will be graded on whether or not you get the correct four numbers for each of these perplexities.

Just run the following cell, which will evaluate the perplexities and print them out. No need to modify the code.

In [ ]:
hf_perp = compute_perplexity(huck_finn_text, emma_lm)
ct_perp = compute_perplexity(ct_yankee_text, emma_lm)
pp_perp = compute_perplexity(pride_and_prejudice_text, emma_lm)
em_perp = compute_perplexity(emma_text, emma_lm)

print("Perplexity on Huckleberry Finn: %.2f" % hf_perp)
print("Perplexity on Connecticut Yankee: %.2f" % ct_perp)
print("Perplexity on Pride and Prejudice: %.2f" % pp_perp)
print("Perplexity on Emma: %.2f" % em_perp)

### Problem 1.4

Now, interpret your results above. Explain the meaning of perplexity for a character-based language model. Which book has the lowest perpexity? Why is this? Which book has the second smallest perplexity? Does this make sense? Explain.



**[Your answer here]**

### Problem 1.5

Next, compute the perplexity of the "Huckleberry Finn" language model on each of the four books. Comment on your findings.

In [ ]:
hf_perp = # Your code here
ct_perp = # Your code here
pp_perp = # Your code here
em_perp = # Your code here

print("Perplexity on Huckleberry Finn: %.2f" % hf_perp)
print("Perplexity on Connecticut Yankee: %.2f" % ct_perp)
print("Perplexity on Pride and Prejudice: %.2f" % pp_perp)
print("Perplexity on Emma: %.2f" % em_perp)

Comment on your findings.

**[Your comment here]**

### Problem 1.6
Finally, you will explore the optimal choice of weights assigned to your unigram, bigram, trigram, and four-gram models. Explore the weights provided in the question, fill in the codes, and comment on the finding.

 Recall that to set the weights on the language model `lm` you can use a function like `lm.set_weights([.25, .25, .25, .25])`.

1. Evaluate the perplexity of the "Pride and Prejudice" text using the following four configurations:

- Heavily Unigram: [0.7, 0.1, 0.1, 0.1]

- Uniform Split: [0.25, 0.25, 0.25, 0.25]

- Increasing Complexity: [0.1, 0.2, 0.3, 0.4]

- Extreme Four-gram: [0.01, 0, 0, 0.99]

Which weight set produced the lowest (best) perplexity? Do these weights make sense to you?

In [ ]:
emma_lm.set_weights([0.7,0.1,0.1,0.1])
w1_perp = compute_perplexity(pride_and_prejudice_text, emma_lm)

emma_lm.set_weights([0.25,0.25,0.25,0.25])
w2_perp = compute_perplexity(pride_and_prejudice_text, emma_lm)

emma_lm.set_weights([0.1,0.2,0.3,0.4])
w3_perp = compute_perplexity(pride_and_prejudice_text, emma_lm)

emma_lm.set_weights([0.01,0,0,0.99])
w4_perp = compute_perplexity(pride_and_prejudice_text, emma_lm)

print("Perplexity using weights [0.7,0.1,0.1,0.1]: %.2f" % w1_perp)
print("Perplexity using weights [0.25,0.25,0.25,0.25]: %.2f" % w2_perp)
print("Perplexity using weights [0.1,0.2,0.3,0.4]: %.2f" % w3_perp)
print("Perplexity using weights [0.01,0,0,0.99]: %.2f" % w4_perp)

Comment your findings.

**[Your comment here]**



2. Using the same weight configurations as above, find the weight set that produces the lowest perplexity of **Huckleberry Finn** using the **Emma** model. First, finish the code. Second, comment on the following: How do the weights you find compare to those you found above? Can you explain intuitively why they are different?

In [ ]:
# Your code here
...
print("Perplexity using weights [0.7,0.1,0.1,0.1]: %.2f" % ...)
print("Perplexity using weights [0.25,0.25,0.25,0.25]: %.2f" % ...)
print("Perplexity using weights [0.1,0.2,0.3,0.4]: %.2f" % ...)
print("Perplexity using weights [0.01,0,0,0.99]: %.2f" % ...)

Comment your findings.

**[Your comment here]**

### Problem 2: Word embedding experiments (15 points)

In this problem you will run experiments on word embeddings using two different algorithms or configurations: (1) word2vec embeddings trained on the "text8" Wikipedia corpus (2) GloVe embeddings pre-trained on a much larger corpus.

The text8 data are described here: http://mattmahoney.net/dc/textdata.html. The text8 file is a 100MB excerpt of Wikipedia. This small dataset is sufficient for our exploratory purposes, but note that it is far too small for any real application.
In the next few parts of this problem, you will construct word embeddings from the Wikipedia data.

`word2Vec` is a popular word embedding method. The following code will construct 100 dimensional embeddings on the text8 data.

Note: If you got the error message "unexpected keyword argument 'size'", it is because you are using a later version of gensim. You can solve the problem by using vector_size instead of size.

In [ ]:
# !pip install gensim # if using colab and get error message "No module named 'gensim'", uncomment this line
import gensim
from gensim.models import word2vec
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', \
                    level=logging.INFO)
sentences = word2vec.Text8Corpus('https://raw.githubusercontent.com/YData123/sds265-sp26/main/assignments/assn4/text8')

In [ ]:
# this block takes time to run
model = word2vec.Word2Vec(sentences, vector_size=100, window=10, min_count=10)

The dictionary `model.wv`, keyed by words (as strings), has values which are the word embeddings (as numpy arrays).

We will also work with pre-trained GloVe embeddings. These embeddings were trained on a large corpus containing 6 billion tokens. You can load these embedding vectors using this code:

In [ ]:
import gensim.downloader as gdl
glove = gdl.load("glove-wiki-gigaword-100")

Here is a sample evaluation: puppy is to dog as what is to kitten?

In [ ]:
glove.most_similar(positive = ['dog', 'kitten'], negative = ['puppy'])[0]

###  Experiments

Conduct the following experiments with **both sets of embeddings**: the local word2vec embeddings, and the pre-trained GloVe embeddings. Based on the available memory on your computer, you may need to perform the experiments for each set of embeddings in a fresh Python session. Comment on the qualitative differences in the results for each of the embeddings.


#### 2.1 Find the closest words

For each of the following words, find the 5 closest words in the embedding space. Here, "closest" means in terms of cosine similarity. See the gensim documentation, and use the **most_similar** function.

          yale, physics, dessert, einstein, algebra, fish


In [ ]:
words = ['yale','physics','dessert','einstein','algebra','fish']
w2v_results = [model.wv.most_similar(i, topn=5) for i in words]
glove_results = # Your code here

In [ ]:
print(w2v_results)
print(glove_results)

Comment on your findings.

**[Your comment here]**

#### 2.2 Complete analogies

A surprising consequence of some word embedding methods is that they can be used to resolve analogies, like

                   france :  paris ::  england :  ?
You can "solve" this analogy by computing the nearest embedding vector to $v$, where $v = v_{paris} − v_{france} + v_{england}$.

Now, solve the following analogies with **both sets** of word embeddings and report your results:

                   france :  paris ::  england :  ?
                   france :  paris ::  germany :  ?
                     queen :  woman ::  king :  ?                    


1. france :  paris ::  england :  ?

In [ ]:
w2v_results = model.wv.most_similar(positive=['paris', 'england'], negative=['france'], topn=1)
glove_results = # Your code here
print(w2v_results)
print(glove_results)

2. france :  paris ::  germany :  ?

In [ ]:
w2v_results = # Your code here
glove_results = # Your code here
print(w2v_results)
print(glove_results)

3. queen :  woman ::  king :  ?

In [ ]:
w2v_results = # Your code here
glove_results = # Your code here
print(w2v_results)
print(glove_results)

Comment on your findings.

**[Your comment here]**

#### 2.3 Visualize embeddings

Use the t-SNE dimensionality reduction technique to visualize the embeddings in two dimensions. The sample code below will perform the t-SNE method on a subset of the vocabulary. You can start with this code and modify it to construct visualizations of the embeddings, or start with your own version of t-SNE.

The code provided shows the relative positions of the words conservative, liberal, elephant, and donkey based on GloVe embeddings. For your local embeddings, you may use similar code to visualize the embeddings.

Find an example that produces expected results and an example that produces surprising results for each of the two language models. Comment on the reason you think that you see surprising behavior here.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


# this functions computes and displays the 2-d t-SNE maps for a subset of the embedding vectors
# and displays them together with the points for a set of input words.

def display_tsne_neighborhood(model, input_word, nsample=1000, size1=2, size2=10, offset=5):

    arr = np.empty((0,100), dtype='f')
    word_label = input_word

    # add the vector for each of the closest words to the array
    for w in range(len(input_word)):
        if 'word2vec' in str(model.__class__):
            arr = np.append(arr, np.array([model.wv.get_vector(input_word[w])]), axis=0)
        else:
            arr = np.append(arr, np.array([model[input_word[w]]]), axis=0)

    if 'word2vec' in str(model.__class__):
        voc = [w for w in model.wv.index_to_key]
        wrds = np.random.choice(range(len(voc)), size=nsample, replace=False)
        for w in wrds:
            wrd_vector = model.wv.get_vector(voc[w])
            arr = np.append(arr, np.array([wrd_vector]), axis=0)
    else:
        voc = [w for w in model.key_to_index]
        wrds = np.random.choice(range(len(voc)), size=nsample, replace=False)
        for w in wrds:
            wrd_vector = model[voc[w]]
            arr = np.append(arr, np.array([wrd_vector]), axis=0)

    # find tsne coords for 2 dimensions
    tsne = TSNE(n_components=2, random_state=0)
    np.set_printoptions(suppress=True)
    Y = tsne.fit_transform(arr)

    x_coord = Y[:, 0]
    y_coord = Y[:, 1]
    # display scatter plot
    size=2
    plt.scatter(x_coord, y_coord, s=size1)
    plt.scatter(x_coord[0:len(input_word)], y_coord[0:len(input_word)],s=size2)

    # label the input words
    for w in range(len(input_word)):
        plt.annotate(input_word[w], xy=(x_coord[w],y_coord[w]), \
                     xytext=(w*offset,w*offset), textcoords='offset points')
    plt.show()

In [ ]:
%matplotlib inline
plt.figure(figsize=(10,10))
display_tsne_neighborhood(glove, input_word = ['conservative', 'liberal', 'donkey', 'elephant'])

In [ ]:
# Your code here for an expected example
...

In [ ]:
# Your code here for a surprising example
...

### Problem  3: Experiments with Musician Embeddings (15 points)


In this problem, we will use a collection of playlists obtained from [last.fm](http://last.fm). We treat each playlist as a document, and each artist in the playlist as a word. By feeding this dataset to word2vec, we will be able to learn artist embeddings.

#### Artist Embeddings

The following experiments will be done with the playlist data file `playlists.txt`. Each line in this file is a playlist. The integers on each line are unique artist identifiers, indicating which artists were in each playlist. The artists are in `artists.txt`.

The code below constructs artist embeddings with word2vec. The artist names are mapped to id numbers in the playlists; the code maps them back to display the names.

In [ ]:
import gensim
from gensim.models import word2vec
import logging

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
playlists = word2vec.LineSentence('https://raw.githubusercontent.com/YData123/sds265-sp26/main/assignments/assn4/playlists.txt')
music_model = word2vec.Word2Vec(playlists, vector_size=64, window=100, min_count=10)
# use size rather than vector_size in gensim version 3.8

In [ ]:
music_model.wv['299']

In [ ]:
from urllib.request import urlopen

artist = []
file = urlopen('https://raw.githubusercontent.com/YData123/sds265-sp26/main/assignments/assn4/artists.txt')
for line in file:
    art = line.decode("utf-8")
    artist.append(art.strip())

artist[0:10]

In [ ]:
id2name = {}
name2id = {}
for w in range(len(artist)):
    id2name["%s" % w] = artist[w]
    name2id[artist[w]] = "%s" % w

id2name[name2id['Elton John']]

#### 3.1 Similar artists

Find the 5 closest artist embedding vectors to the artists "The Beatles", "Lady Gaga", and "Nirvana". Comment on the results.

In [ ]:
# Don't change this function
def similar_artists(model, artist, n=5):
    id = name2id[artist]
    out = model.wv.most_similar(id, topn=n)

    print("artists similar to '%s'\n" % artist)
    for i in range(n) :
        name = id2name[out[i][0]]
        print("\t%s" % name)

similar_artists(music_model, 'Aerosmith')

In [ ]:
# Your code here
...

Comment on your findings.

** Your comment here **

#### 3.2 Visualize embeddings

Use the t-SNE dimensionality reduction technique to visualize the artist embeddings. After running t-SNE on the artist embeddings, try visualizing "The Temptations" and "The Supremes" together.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


# this functions computes and displays the 2-d t-SNE maps for a subset of the embedding vectors
# and displays them together with the points for a set of input words.

def display_tsne_artists(model, artists, nsample=1000, size1=2, size2=10, offset=5):

    arr = np.empty((0,64), dtype='f')

    # add the vector for each of the closest words to the array
    for a in range(len(artists)):
        id = name2id[artists[a]]
        arr = np.append(arr, np.array([model.wv.get_vector(id)]), axis=0)

    voc = [w for w in model.wv.index_to_key]
    ids = np.random.choice(range(len(voc)), size=nsample, replace=False)
    for w in ids:
        wrd_vector = model.wv.get_vector(voc[w])
        arr = np.append(arr, np.array([wrd_vector]), axis=0)

    # find tsne coords for 2 dimensions
    tsne = TSNE(n_components=2, random_state=0)
    np.set_printoptions(suppress=True)
    Y = tsne.fit_transform(arr)

    x_coord = Y[:, 0]
    y_coord = Y[:, 1]
    # display scatter plot
    size=2
    plt.scatter(x_coord, y_coord, s=size1)
    plt.scatter(x_coord[0:len(artists)], y_coord[0:len(artists)],s=size2)

    # label the input words
    for w in range(len(artists)):
        plt.annotate(artists[w], xy=(x_coord[w],y_coord[w]), \
                     xytext=(w*offset,w*offset), textcoords='offset points')
    plt.show()

In [ ]:
# Your code here
...

Comment on your findings.

**[Your comment here]**

Pick another example that you think is interesting and comment on your findings.

In [ ]:
# Your code and comment here

### Problem 4 Exploring MLP Architecture and Overfitting using Tensorflow Playground (15 points)
In this problem, you will use [TensorFlow Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.97904&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false) to build intuition for how Multi-Layer Perceptrons (MLPs) work. Specifically, you will investigate how model complexity (layers, nodes, features) and training parameters affect a model's ability to generalize, and also demonstrate the concept of "overfitting."

#### 4.1 Generate two datasets and models
Step 1: Generate two datasets

First, you will design **two distinct datasets** (Dataset 1 and Dataset 2) using the "Data" column on the far left of the playground.

For each dataset, report the following design parameters:
* Starting Template: Choose one of the four patterns (Circles, Exclusive OR/Grid, Clusters, or Spirals).
* Ratio of Training to Test Data: (e.g., 50%)
* Noise: (e.g., 10-20% is usually good for forcing a model to overfit)
* Batch Size: (e.g., 10)

Step 2: Model Design and Training

You will design and train a total of four distinct models. You may adjust the input features ($X_1$, $X_2$, $X_1^2$, etc.), the number of hidden layers, the number of neurons per layer, the learning rate, and the regularization (L1/L2 or none), etc.

For Dataset 1:

* Model A (The Overfitter): Train this model on Dataset 1 with a hyperparameter setting where the model significantly overfits the data (e.g., low training loss but high testing loss).
* Model B (The Good Fit): Train this model on Dataset 1 using a different hyperparameter setting that captures the true underlying pattern of the same dataset without memorizing the noise. The Test Loss and Training Loss should be relatively close and low.

For Dataset 2:

* Model C (The Overfitter): Train this model on Dataset 2, again choosing hyperparameters where the model significantly overfits the data.
* Model D (The Good Fit): Train this model on Dataset 2 using hyperparameters that allow it to generalize well.

Allow each of your four models to train until the loss curves stabilize. **Take a screenshot of all four models and write down the parameters.** Ensure your screenshots capture the Data parameters, the Network Architecture, the final Output graph (decision boundary), and the "Test loss" vs "Training loss" metrics at the top right.

Screenshot for Model A

[here]

Screenshot for Model B

[here]

Screenshot for Model C

[here]

Screenshot for Model D

[here]

#### 4.2 Discuss your results

Submit a brief write-up analyzing your results. For each model, answer the following questions:

* What specific changes did you make to the features, hidden layers, or number of nodes to transition from an overfit model to a well-fitted model? Why did changing these parameters help?

* Look at the background color map in the final output column. How does the decision boundary of your overfitted model visually differ from your well-fitted model in terms of smoothness, complexity, and how it treats outlier "noisy" dots?

* If applicable: Look at the "Features" column. If you provide the model with complex, non-linear features (like $X_1^2$, $X_2^2$, or $\sin(X_1)$), how does that change the number of hidden layers or neurons required to get a good fit?



**[Your comment here]**